# 02 — Data Preparation, Cleaning, Imputation & Baseline Model

This notebook covers the **Data Preparation** phase of the CRISP-DM workflow and the initial baseline model check, matching `split_training_test_data()`, `reduce_features()`, `treat_outliers()`, `knn_imputation()`, and `evaluate_baseline()` in `src/secom_analysis.py` and `src/plots.py`.

### Leakage-Prevention Principle
- The dataset is split **80/20 using stratified sampling (`random_state=42`) before any preprocessing**.
- The 20% test set (`X_test`, `y_test`) is locked away immediately after splitting and **never touched** during feature reduction, outlier boundary calculation, scaling, or kNN imputation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

base = "https://archive.ics.uci.edu/ml/machine-learning-databases/secom/"
X = pd.read_csv(base + "secom.data", sep=" ", header=None)
labels = pd.read_csv(base + "secom_labels.data", sep=" ", header=None, names=["target", "timestamp"])
y = labels["target"]
X.columns = [f"sensor_{i}" for i in range(X.shape[1])]

--- 
## Stratified 80/20 Train/Test Split

- **Why Stratified Sampling (`stratify=y`)?** Because only `6.64%` (`104` of `1,567`) wafers are defective, simple random sampling could by chance place disproportionately few or many defective wafers into the 20% test partition, skewing final sensitivity/specificity estimates. Stratified sampling partitions the Pass (`-1`) and Fail (`1`) strata independently to guarantee identical class proportions (`~93.4%` Pass / `~6.6%` Fail) across Original, Training, and Test sets.
- **Why `random_state=42`?** Locks the pseudo-random number generator so the exact same `1,253` training wafers and `314` test wafers are selected on every run, even when downloaded dynamically at runtime.

In [ ]:
def split_training_test_data(X, y):
    print("Performing Stratified 80/20 Split...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print(f"Training set: {len(X_train)} wafers")
    print(f"Test set: {len(X_test)} wafers")

    comparison = pd.DataFrame({
        "Original (%)": (y.value_counts(normalize=True) * 100).round(2),
        "Training (%)": (y_train.value_counts(normalize=True) * 100).round(2),
        "Test (%)": (y_test.value_counts(normalize=True) * 100).round(2),
        "Original Count": y.value_counts(),
        "Training Count": y_train.value_counts(),
        "Test Count": y_test.value_counts(),
    })
    print("\nTarget Distribution Before and After Split:")
    print(comparison.to_string())
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = split_training_test_data(X, y)

--- 
## Rough Feature Reduction (Calculated Exclusively on `X_train`)

All reduction checks are computed strictly on `X_train` (`1,253` rows):
1. **Constant / Zero-Variance Features (`var == 0`)**: Sensors with zero variance record the exact same value for every training wafer and carry zero information for separating Pass vs. Fail (`116` sensors dropped → `474` remaining).
2. **Duplicate Rows & Duplicate Columns**: Check `X_train_reduced.duplicated().sum()` (`0` duplicate rows) and `X_train_reduced.T.duplicated()` (`0` duplicate columns after constant removal).
3. **Low-Variance ("Low Vola") Analysis (No Action)**: We inspect non-zero sensor variances using a log-scale Histogram and Pareto chart. Because sensors are unscaled at this stage, small numeric variance reflects measurement units (e.g. millivolts vs. degrees Celsius) rather than lack of predictive power. Decision: **Keep all (`0` dropped)**.
4. **Missingness Analysis & `>60%` Threshold**:
   - We inspect the Missingness Histogram and Pareto chart across thresholds (`0%`–`100%`).
   - We also check missingness grouped by class (`Pass (-1)` vs. `Fail (1)`) on the high-missing columns and overall (`26.53` vs. `24.73` missing sensors/wafer) to verify missingness is random rather than a defect indicator.
   - **Threshold Decision**: Using a conservative `60%` threshold (compared to `55%` in earlier literature), we drop the `24` sensors exceeding `60%` missing values, leaving **`450` sensors**.

In [ ]:
variances = X_train.var(numeric_only=True)
constant_cols = variances[variances == 0].index.tolist()
X_train_reduced = X_train.drop(columns=constant_cols)
print(f"Dropped {len(constant_cols)} constant sensors. (Remaining: {X_train_reduced.shape[1]})")

print(f"Duplicate rows in X_train: {X_train_reduced.duplicated().sum()}")
duplicate_cols = X_train_reduced.columns[X_train_reduced.T.duplicated()].tolist()
X_train_reduced = X_train_reduced.drop(columns=duplicate_cols)
print(f"Dropped {len(duplicate_cols)} duplicate sensors.")

non_zero_var = X_train_reduced.var(numeric_only=True)
low_var_subset = non_zero_var[non_zero_var < 0.01]
print(f"Sensors with low variance (0 < var < 0.01): {len(low_var_subset)} (Action: Keep all)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log10(non_zero_var), bins=30, edgecolor="black")
axes[0].axvline(np.log10(0.01), color="red", linestyle="--", label="var = 0.01")
axes[0].set_title("Histogram of Sensor Variances (log10 scale)")
axes[0].set_xlabel("log10(Variance)")
axes[0].set_ylabel("Number of Sensors")
axes[0].legend()

var_thresholds = np.logspace(-6, 0, 25)
removed_if_cut = [(non_zero_var < t).sum() for t in var_thresholds]
axes[1].plot(var_thresholds, removed_if_cut, marker="o")
axes[1].set_xscale("log")
axes[1].set_title("Pareto Chart: Sensors Removed vs. Variance Threshold")
axes[1].set_xlabel("Variance Threshold (log scale)")
axes[1].set_ylabel("Number of Sensors Removed")
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
missing_pct = X_train_reduced.isna().mean() * 100
high_missing_cols = missing_pct[missing_pct > 60].index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(missing_pct, bins=20, edgecolor="black")
axes[0].axvline(60, color="red", linestyle="--", label="60% Threshold")
axes[0].set_title("Histogram of Missing Values Across Sensors")
axes[0].set_xlabel("Missing Values (%)")
axes[0].set_ylabel("Number of Sensors")
axes[0].legend()

thresholds = np.arange(0, 101, 5)
removed_counts = [(missing_pct > t).sum() for t in thresholds]
axes[1].plot(thresholds, removed_counts, marker="o")
axes[1].axvline(60, color="red", linestyle="--", label="60% Threshold")
axes[1].set_title("Pareto Chart: Sensors Removed vs. Missing Threshold")
axes[1].set_xlabel("Missingness Threshold (%)")
axes[1].set_ylabel("Number of Sensors Removed")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
fig.tight_layout()
plt.show()

print("\nMissingness Breakdown for High-Missing Columns (>60%):")
for col in high_missing_cols:
    col_missing = X_train_reduced[col].isna().groupby(y_train).mean() * 100
    print(f"{col} -> Pass (-1): {col_missing[-1]:.1f}% | Fail (1): {col_missing[1]:.1f}%")

missing_by_class = X_train_reduced.isna().sum(axis=1).groupby(y_train).mean()
print("\nAverage missing sensors per wafer (X_train):")
print(f"Pass (-1): {missing_by_class[-1]:.2f}")
print(f"Fail (1): {missing_by_class[1]:.2f}")

X_train_reduced = X_train_reduced.drop(columns=high_missing_cols)
print(f"\nDropped {len(high_missing_cols)} sensors with >60% missing data.")
print(f"Sensors remaining: {X_train_reduced.shape[1]}")

--- 
## Outlier Identification & Treatment ($3\sigma$ Rule)

- **Why $3\sigma$ instead of IQR?** Manufacturing sensor distributions naturally exhibit operational spread that IQR (`1.5 × IQR`) over-flags. The $3\sigma$ rule ($\mu \pm 3\sigma$, calculated per sensor on `X_train` only) targets only severe spikes.
- **Why Capping / Winsorizing (`.clip()`) instead of Removal (`.mask()`) or Do Nothing?**
  - *Do Nothing*: Leaves extreme spikes that distort Euclidean distances during downstream `MinMaxScaler` and `KNNImputer` steps.
  - *Removal (Masking to `NaN`)*: Injects `7,540` brand-new `NaN` gaps into `X_train` that would have to be artificially reconstructed during imputation.
  - *Capping at $\mu \pm 3\sigma$ (Chosen)*: Bounds all `7,540` extreme readings in `X_train` at the training $\pm 3\sigma$ limits without losing a single observation or creating new missing values.

In [ ]:
def treat_outliers(X_train):
    col_mean = X_train.mean(numeric_only=True)
    col_std = X_train.std(numeric_only=True)
    lower_bound = col_mean - 3 * col_std
    upper_bound = col_mean + 3 * col_std

    outlier_mask_train = (X_train < lower_bound) | (X_train > upper_bound)
    total_flagged_train = outlier_mask_train.sum().sum()
    print(f"Total outliers flagged in training set: {total_flagged_train}")

    X_train_cleaned = X_train.clip(lower=lower_bound, upper=upper_bound, axis=1)
    return X_train_cleaned


X_train_cleaned = treat_outliers(X_train_reduced)

--- 
## Feature Scaling & kNN Imputation ($k=5$) + Baseline Model

- **Why Mean Imputation is Avoided**: Filling gaps with a constant column mean artificially collapses feature variance and erases inter-sensor relationships.
- **Why Pre-Scale with `MinMaxScaler` Before `KNNImputer(n_neighbors=5)`**: `KNNImputer` uses Euclidean distance across all 450 sensors to locate the 5 nearest neighbor wafers. Without scaling, sensors measured in thousands dominate the distance metric over sensors measured in fractions. Because extreme outliers were already bounded at $\pm 3\sigma$, `MinMaxScaler` cleanly maps every training sensor to `[0, 1]` before fitting `KNNImputer(n_neighbors=5)` on `X_train`.
- **Baseline Random Forest Check**: Fitting `RandomForestClassifier(random_state=42)` on `X_train_imputed, y_train` yields `100.00%` training accuracy — demonstrating memorization on training data and motivating subsequent feature selection, class balancing, and held-out test evaluation.

In [ ]:
def knn_imputation(X_train):
    print("Performing KNN Imputation...")
    scaler = MinMaxScaler()
    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
    )

    imputer = KNNImputer(n_neighbors=5)
    X_train_imputed = pd.DataFrame(
        imputer.fit_transform(X_train_scaled),
        columns=X_train.columns,
        index=X_train.index,
    )

    print(f"Missing values remaining in Train: {X_train_imputed.isna().sum().sum()}")
    return X_train_imputed


def evaluate_baseline(X_train, y_train):
    print("\nBaseline Random Forest (Pre-Feature Selection)...")
    rf_baseline = RandomForestClassifier(random_state=42)
    rf_baseline.fit(X_train, y_train)

    train_accuracy = rf_baseline.score(X_train, y_train) * 100
    print(f"Baseline Training Accuracy: {train_accuracy:.2f}%")
    return rf_baseline


X_train_imputed = knn_imputation(X_train_cleaned)
rf_baseline = evaluate_baseline(X_train_imputed, y_train)

--- 
## Step Impact Protocol (SIP) Table — Data Preparation

| Pipeline Step | Options / Approaches Available | Pros & Cons of Each Method | Decision Made | Consequence (Before → After) |
| :--- | :--- | :--- | :--- | :--- |
| **1. Train/Test Split** | • Simple Random Split<br>• Stratified 80/20 Split (`stratify=y`) | • **Random**: Risk of uneven defect % in test set due to small minority count (`104` total fails).<br>• **Stratified**: Guarantees identical `93.4% / 6.6%` Pass/Fail ratio in both partitions. | **Stratified 80/20 Split** (`random_state=42`, test set locked away) | **Before**: `1,567 × 590`<br>**After**: `X_train` = `1,253 × 590` (`93.38%` Pass, `6.62%` Fail); `X_test` = `314 × 590` (`93.31%` Pass, `6.69%` Fail) |
| **2. Zero-Variance Removal** | • Keep constant columns<br>• Drop columns with `var == 0` on `X_train` | • **Keep**: Adds redundant dimensions with zero predictive signal.<br>• **Drop**: Removes completely uninformative sensors with zero information loss. | **Drop `var == 0` columns** | **Before**: `590` sensors<br>**After**: `474` sensors (`116` constant sensors removed) |
| **3. Duplicate & Low-Vola Check** | • Drop duplicate columns (`X_train.T.duplicated()`)<br>• Drop low-variance (`var < 0.01`) vs. Keep | • **Duplicates**: Redundant identical signals.<br>• **Low-Vola**: Unscaled sensors have small variance due to physical units; dropping now risks losing valid signals. | **Drop exact duplicates; Keep all low-vola columns** (*No action on low-vola*) | **Before**: `474` sensors<br>**After**: `474` sensors (`0` exact duplicates found; `97` low-vola sensors retained) |
| **4. High-Missingness Removal** | • `55%` threshold<br>• `60%` threshold<br>• Keep all and impute | • **`55%`**: More aggressive early feature loss.<br>• **`60%`**: Conservative cut at Pareto elbow; avoids imputing columns that are mostly empty. | **Drop sensors with `>60%` missing values** on `X_train` | **Before**: `474` sensors<br>**After**: `450` sensors (`24` high-missing sensors removed; `16,284` `NaN`s remain in `X_train`) |
| **5. Outlier Treatment** | • **Option A**: Cap (Winsorize) at $\mu \pm 3\sigma$<br>• **Option B**: Remove / mask as `NaN`<br>• **Option C**: Do nothing | • **Cap**: Bounds extreme spikes without creating new `NaN` gaps.<br>• **Mask**: Creates `7,540` new `NaN`s requiring imputation.<br>• **Do nothing**: Distorts Euclidean distances in kNN. | **Option A: Cap at $\mu \pm 3\sigma$** (computed on `X_train` only) | **Before**: `450` sensors (`7,540` outlier cells flagged in `X_train`)<br>**After**: `450` sensors (`7,540` cells capped at $\pm 3\sigma$; `0` new `NaN`s introduced) |
| **6. Scaling & Imputation** | • Mean imputation<br>• Unscaled `KNNImputer(k=5)`<br>• `MinMaxScaler` + `KNNImputer(k=5)` | • **Mean**: Destroys variance.<br>• **Unscaled kNN**: Large-scale sensors dominate Euclidean neighbor search.<br>• **MinMax + kNN ($k=5$)**: Equal `[0, 1]` sensor weights during neighbor lookup. | **`MinMaxScaler` + `KNNImputer(n_neighbors=5)`** fit strictly on `X_train` | **Before**: `16,284` missing cells in `X_train`<br>**After**: `0` missing cells in `X_train_imputed` (`1,253 × 450`) |